# 01_data.ipynb — Sleep-EDF Expanded: Extraction and Processing

**Source:** `../physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/`  
**Output:** `../data/SC/` (per-subject NPZ) + `../data/metadata.csv`

**What this does:**
- Discovers all night-1 cassette (SC) PSG + hypnogram pairs
- Loads EEG Fpz-Cz, EEG Pz-Oz, EOG horizontal, EMG submental at 100 Hz
- Segments into 30-second epochs, maps R&K stages to AASM (merges stages 3+4 into N3), drops MT and ?
- Flags artifact epochs (EEG peak-to-peak > 150 µV) without removing them
- Saves per-subject `.npz` with keys: `epochs` (n, 4, 3000), `labels` (n,), `artifact_mask` (n,)
- Saves `metadata.csv` with per-subject stage counts and recording stats

In [1]:
import mne
import numpy as np
import pandas as pd
from pathlib import Path
import warnings

mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')

In [2]:
# Paths
SC_DIR  = Path('../physionet.org/files/sleep-edfx/1.0.0/sleep-cassette')
OUT_DIR = Path('../data/SC')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Signal config
CHANNELS      = ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'EMG submental']
SFREQ         = 100          # Hz — resample target
EPOCH_SEC     = 30
EPOCH_SAMPLES = SFREQ * EPOCH_SEC   # 3000 samples per epoch
ARTIFACT_UV   = 150e-6              # 150 µV peak-to-peak threshold (EEG channels only)

# R&K to AASM label mapping  (-1 = drop)
STAGE_MAP = {
    'Sleep stage W': 0,
    'Sleep stage 1': 1,
    'Sleep stage 2': 2,
    'Sleep stage 3': 3,
    'Sleep stage 4': 3,   # merge into N3
    'Sleep stage R': 4,
    'Movement time': -1,
    'Sleep stage ?': -1,
}
STAGE_NAMES = {0: 'Wake', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'REM'}

## Step 1 — Discover night-1 PSG / hypnogram pairs

In [3]:
def find_pairs(sc_dir: Path) -> list[dict]:
    """
    Match *-PSG.edf files with their hypnogram counterparts.
    Returns night-1 only, sorted by subject ID.

    Filename convention:
      PSG:       SC4{SS}{N}{X}0-PSG.edf         e.g. SC4001E0-PSG.edf
      Hypnogram: SC4{SS}{N}{X}{C}-Hypnogram.edf  e.g. SC4001EC-Hypnogram.edf
      SS=subject (2 digits), N=night, X=series letter, C=scorer code
    PSG and hypnogram share only the first 7 chars (e.g. 'SC4001E').
    """
    pairs = []
    for psg in sorted(sc_dir.glob('*-PSG.edf')):
        stem    = psg.stem.replace('-PSG', '')   # e.g. 'SC4001E0'
        subject = int(stem[3:5])                 # chars 3-4: subject ID
        night   = int(stem[5])                   # char 5: night number
        if night != 1:
            continue
        # Hypnogram shares first 7 chars then has an arbitrary scorer letter
        matches = list(sc_dir.glob(f'{stem[:7]}*-Hypnogram.edf'))
        if not matches:
            print(f'  [SKIP] no hypnogram for {psg.name}')
            continue
        pairs.append({'subject': subject, 'psg': psg, 'hypnogram': matches[0]})
    return sorted(pairs, key=lambda x: x['subject'])


pairs = find_pairs(SC_DIR)
# Subject 36 has no night-1 recording in this dataset — expect 152 pairs, not 153
print(f'Found {len(pairs)} night-1 SC recordings')

Found 76 night-1 SC recordings


## Step 2 — Verify channel names (run once)

Confirm the channel names in the raw files match `CHANNELS` before batch processing.

In [4]:
raw0 = mne.io.read_raw_edf(pairs[0]['psg'], preload=False, stim_channel=None)
print('Available channels:', raw0.info['ch_names'])
print('Native sampling rate:', raw0.info['sfreq'], 'Hz')
print('Duration:', round(raw0.times[-1] / 3600, 2), 'hours')

missing = [c for c in CHANNELS if c not in raw0.info['ch_names']]
if missing:
    print(f'\n[WARNING] Channels not found: {missing}')
    print('Update CHANNELS to match the names above.')
else:
    print('\nAll target channels confirmed present.')

Available channels: ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker']
Native sampling rate: 100.0 Hz
Duration: 22.08 hours

All target channels confirmed present.


## Step 3 — Processing pipeline

In [5]:
def process_recording(psg_path: Path, hyp_path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Load one overnight recording and return epoch arrays.

    Returns
    -------
    epochs        : float32 (n, 4, 3000)  — channel order matches CHANNELS
    labels        : int8   (n,)           — 0=Wake 1=N1 2=N2 3=N3 4=REM
    artifact_mask : bool   (n,)           — True where EEG ptp > 150 µV
    """
    # Load and resample
    raw = mne.io.read_raw_edf(psg_path, include=CHANNELS, preload=True, stim_channel=None)
    if raw.info['sfreq'] != SFREQ:
        raw.resample(SFREQ, npad='auto')

    data = raw.get_data()          # (4, n_total_samples)  float64
    n_total = data.shape[1]

    annots = mne.read_annotations(hyp_path)

    epochs, labels = [], []
    for ann in annots:
        label = STAGE_MAP.get(ann['description'], -1)
        if label == -1:
            continue
        start = int(round(ann['onset'] * SFREQ))
        stop  = start + EPOCH_SAMPLES
        if stop > n_total:          # last partial epoch — skip
            continue
        epochs.append(data[:, start:stop])
        labels.append(label)

    epochs = np.array(epochs, dtype=np.float32)    # (n, 4, 3000)
    labels = np.array(labels,  dtype=np.int8)       # (n,)

    # Artifact flag: EEG channels only (indices 0 and 1)
    ptp = np.ptp(epochs[:, :2, :], axis=-1).max(axis=1)   # (n,) max ptp across EEG channels
    artifact_mask = ptp > ARTIFACT_UV                       # (n,) bool

    return epochs, labels, artifact_mask

## Step 4 — Batch process all recordings

In [6]:
records = []

for i, p in enumerate(pairs):
    sid = p['subject']
    out_path = OUT_DIR / f'SC{sid:02d}.npz'

    if out_path.exists():
        # Resume-safe: skip already-processed subjects
        arr = np.load(out_path)
        labels = arr['labels']
    else:
        try:
            epochs, labels, artifact_mask = process_recording(p['psg'], p['hypnogram'])
        except Exception as e:
            print(f'  [ERROR] subject {sid:02d}: {e}')
            continue
        np.savez_compressed(out_path, epochs=epochs, labels=labels, artifact_mask=artifact_mask)

    # Collect metadata regardless of whether we just processed or loaded from cache
    counts = {STAGE_NAMES[s]: int((labels == s).sum()) for s in range(5)}
    records.append({
        'subject': sid,
        'n_epochs': len(labels),
        'n_artifact': int(np.load(out_path)['artifact_mask'].sum()) if out_path.exists() else 0,
        **counts,
    })

    print(f'[{i+1:02d}/{len(pairs)}] subject {sid:02d} — '
          f'{len(labels)} epochs | '
          + ' '.join(f'{k}:{v}' for k, v in counts.items()))

print(f'\nDone. {len(records)} subjects processed.')

[01/76] subject 00 — 153 epochs | Wake:12 N1:24 N2:40 N3:71 REM:6
[02/76] subject 01 — 125 epochs | Wake:15 N1:33 N2:40 N3:29 REM:8
[03/76] subject 02 — 160 epochs | Wake:9 N1:22 N2:56 N3:58 REM:15
[04/76] subject 03 — 118 epochs | Wake:14 N1:30 N2:42 N3:22 REM:10
[05/76] subject 04 — 159 epochs | Wake:30 N1:43 N2:47 N3:15 REM:24
[06/76] subject 05 — 129 epochs | Wake:12 N1:19 N2:45 N3:50 REM:3
[07/76] subject 06 — 77 epochs | Wake:13 N1:18 N2:27 N3:15 REM:4
[08/76] subject 07 — 115 epochs | Wake:5 N1:23 N2:33 N3:43 REM:11
[09/76] subject 08 — 141 epochs | Wake:9 N1:14 N2:42 N3:72 REM:4
[10/76] subject 09 — 137 epochs | Wake:8 N1:10 N2:54 N3:48 REM:17
[11/76] subject 10 — 61 epochs | Wake:14 N1:15 N2:22 N3:6 REM:4
[12/76] subject 11 — 124 epochs | Wake:6 N1:5 N2:50 N3:57 REM:6
[13/76] subject 12 — 127 epochs | Wake:19 N1:33 N2:38 N3:29 REM:8
[14/76] subject 13 — 146 epochs | Wake:19 N1:25 N2:37 N3:53 REM:12
[15/76] subject 14 — 121 epochs | Wake:20 N1:16 N2:30 N3:48 REM:7
[16/76] subje

## Step 5 — Save metadata and verify

In [7]:
meta = pd.DataFrame(records).sort_values('subject').reset_index(drop=True)
meta.to_csv('../data/metadata.csv', index=False)

print('=== Dataset Summary ===')
print(f'Subjects       : {len(meta)}')
print(f'Total epochs   : {meta["n_epochs"].sum():,}')
print(f'Artifact epochs: {meta["n_artifact"].sum():,} '
      f'({100 * meta["n_artifact"].sum() / meta["n_epochs"].sum():.1f}%)')
print()
print('Stage distribution (% of total clean epochs):')
total_clean = meta['n_epochs'].sum() - meta['n_artifact'].sum()
for stage in ['Wake', 'N1', 'N2', 'N3', 'REM']:
    n = meta[stage].sum()
    print(f'  {stage:5s}: {n:6,}  ({100*n/meta["n_epochs"].sum():.1f}%)')

print()
print('Output files written to ../data/SC/')
print('Metadata written to ../data/metadata.csv')

=== Dataset Summary ===
Subjects       : 76
Total epochs   : 10,764
Artifact epochs: 4,086 (38.0%)

Stage distribution (% of total clean epochs):
  Wake :  1,793  (16.7%)
  N1   :  2,783  (25.9%)
  N2   :  3,163  (29.4%)
  N3   :  2,339  (21.7%)
  REM  :    686  (6.4%)

Output files written to ../data/SC/
Metadata written to ../data/metadata.csv


## Loading convention for downstream notebooks

```python
import numpy as np
from pathlib import Path

def load_subject(sid: int, data_dir: Path = Path('../data/SC')) -> tuple:
    arr = np.load(data_dir / f'SC{sid:02d}.npz')
    return arr['epochs'], arr['labels'], arr['artifact_mask']

# Example: load subject 0, exclude artifact epochs
epochs, labels, artifact_mask = load_subject(0)
clean = ~artifact_mask
X, y = epochs[clean], labels[clean]   # (n_clean, 4, 3000), (n_clean,)
```

Channel order in axis 1: `[EEG Fpz-Cz, EEG Pz-Oz, EOG horizontal, EMG submental]`  
Label encoding: `0=Wake  1=N1  2=N2  3=N3  4=REM`